# Results charts

This notebook generates figures for the large-scale analysis after scoring the papers.

In [ ]:
from pathlib import Path
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
from matplotlib.ticker import FixedLocator, FormatStrFormatter

ROOT = Path.cwd()
if ROOT.name == 'notebooks':
    ROOT = ROOT.parent
SCORES_CSV = ROOT / 'smart_grobid_repro_scores_3200.csv'
PER_ITEM_CSV = ROOT / 'per_item_accuracy.csv'
FIGURES_DIR = ROOT / 'figures'
FIGURES_DIR.mkdir(exist_ok=True)

VENUE_COLORS = {
    'AAAI': '#4C78A8',
    'ACL': '#2A9D8F',
    'NeurIPS': '#F4A261',
}

METHOD_LABELS = {
    'pymupdf_8b_accuracy': 'PyMuPDF Pipeline\n(Llama 8b)',
    'pymupdf_70b_accuracy': 'PyMuPDF Pipeline\n(Llama 70b)',
    'smart_pymupdf_8b_accuracy': 'Smart PyMuPDF Pipeline\n(Llama 8b)',
    'smart_pymupdf_70b_accuracy': 'Smart PyMuPDF Pipeline\n(Llama 70b)',
    'smart_grobid_8b_accuracy': 'Smart GROBID Pipeline\n(Llama 8b)',
    'smart_grobid_accuracy': 'Smart GROBID Pipeline\n(Llama 70b)',
}

ITEM_LABELS = {
    'dataset_info': 'Dataset info',
    'data_split': 'Data split',
    'hyperparameters': 'Hyperparameters',
    'hardware_info': 'Hardware info',
    'code_repo': 'Code repository',
    'pseudocode': 'Pseudocode',
    'theorems': 'Theorems',
    'proofs': 'Proofs',
    'method_well_described': 'Method described',
}

sns.set_theme(
    style='white',
    context='talk',
    rc={
        'axes.facecolor': 'white',
        'figure.facecolor': 'white',
        'axes.edgecolor': '#111827',
        'axes.labelcolor': '#111827',
        'xtick.color': '#111827',
        'ytick.color': '#111827',
        'text.color': '#111827',
        'axes.grid': False,
        'font.family': 'Arial',
        'font.sans-serif': ['Arial', 'Helvetica', 'DejaVu Sans'],
    },
)


## Mean reproducibility score across venues (2023--2025)

In [ ]:
df = pd.read_csv(SCORES_CSV)
df = df[df['scored'].str.lower() == 'yes'].copy()
df = df[df['conference'].isin(['aaai', 'acl', 'neurips'])]
df = df[df['year'].astype(str).isin(['2023', '2024', '2025'])]
df['conference'] = df['conference'].map({'aaai': 'AAAI', 'acl': 'ACL', 'neurips': 'NeurIPS'})
df['final_score'] = df['final_score'].astype(float)

summary = (
    df.groupby('conference', as_index=False)
    .agg(mean_score=('final_score', 'mean'), n=('final_score', 'size'))
)
summary['conference'] = pd.Categorical(summary['conference'], categories=['AAAI', 'ACL', 'NeurIPS'], ordered=True)
summary = summary.sort_values('conference')
summary


In [ ]:
fig, ax = plt.subplots(figsize=(8.2, 5.2))

sns.barplot(
    data=summary,
    x='conference',
    y='mean_score',
    hue='conference',
    palette=VENUE_COLORS,
    dodge=False,
    width=0.42,
    saturation=0.95,
    ax=ax,
    legend=False,
)

ax.set_title('Mean reproducibility score across venues (2023-2025)', fontsize=16, pad=12, weight='semibold')
ax.set_xlabel('Conference venue', fontsize=12, fontweight='bold')
ax.set_ylabel('Mean normalized reproducibility score', fontsize=12, fontweight='bold')
ax.set_ylim(0, 1.0)
ax.yaxis.set_major_locator(FixedLocator([0.0, 0.5, 1.0]))
ax.yaxis.set_major_formatter(FormatStrFormatter('%.1f'))
ax.minorticks_off()

ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)
ax.spines['left'].set_linewidth(1.0)
ax.spines['bottom'].set_linewidth(1.0)
ax.tick_params(axis='both', which='major', labelsize=11, length=4, width=1.1)
for label in ax.get_xticklabels() + ax.get_yticklabels():
    label.set_fontweight('bold')

for patch, (_, row) in zip(ax.patches, summary.iterrows()):
    x = patch.get_x() + patch.get_width() / 2
    y = patch.get_height()
    ax.text(x, y + 0.015, f"{row['mean_score']:.3f}", ha='center', va='bottom', fontsize=11, weight='semibold')

plt.tight_layout()

png_path = FIGURES_DIR / 'venue_mean_repro_score_2023_2025.png'
svg_path = FIGURES_DIR / 'venue_mean_repro_score_2023_2025.svg'
fig.savefig(png_path, dpi=300, bbox_inches='tight')
fig.savefig(svg_path, bbox_inches='tight')
plt.show()

print(png_path)
print(svg_path)


## Per-item accuracy across pipeline variants

In [ ]:
per_item = pd.read_csv(PER_ITEM_CSV)
accuracy_columns = list(METHOD_LABELS.keys())

heatmap_df = per_item[['item'] + accuracy_columns].copy()
heatmap_df['item'] = heatmap_df['item'].map(ITEM_LABELS)
heatmap_df = heatmap_df.set_index('item')
heatmap_df = heatmap_df.rename(columns=METHOD_LABELS)
heatmap_df = heatmap_df.astype(float)
heatmap_df


In [ ]:
clean_method_labels = {
    'PyMuPDF Pipeline\n(Llama 8b)': 'PyMuPDF 8b',
    'PyMuPDF Pipeline\n(Llama 70b)': 'PyMuPDF 70b',
    'Smart PyMuPDF Pipeline\n(Llama 8b)': 'Smart PyMuPDF 8b',
    'Smart PyMuPDF Pipeline\n(Llama 70b)': 'Smart PyMuPDF 70b',
    'Smart GROBID Pipeline\n(Llama 8b)': 'Smart GROBID 8b',
    'Smart GROBID Pipeline\n(Llama 70b)': 'Smart GROBID 70b',
}

clean_item_labels = {
    'Dataset info': 'Dataset info',
    'Data split': 'Data split',
    'Hyperparameters': 'Hyperparams',
    'Hardware info': 'Hardware',
    'Code repository': 'Code repo',
    'Pseudocode': 'Pseudocode',
    'Theorems': 'Theorems',
    'Proofs': 'Proofs',
    'Method described': 'Method desc.',
}

heatmap_clean = heatmap_df.copy()
heatmap_clean = heatmap_clean.rename(columns=clean_method_labels)
heatmap_clean.index = [clean_item_labels.get(idx, idx) for idx in heatmap_clean.index]

soft_cmap = sns.blend_palette(['#F6F7FB', '#BFD6E6', '#7CA8C7', '#3F7CA8'], as_cmap=True)

fig, ax = plt.subplots(figsize=(10.8, 5.0))
sns.heatmap(
    heatmap_clean,
    cmap=soft_cmap,
    vmin=0.25,
    vmax=1.0,
    annot=False,
    linewidths=0,
    cbar=True,
    cbar_kws={'shrink': 0.88, 'ticks': [0.25, 0.5, 0.75, 1.0]},
    ax=ax,
)

ax.set_title('Per-item accuracy across pipeline variants', fontsize=16, pad=12, weight='semibold')
ax.set_xlabel('Pipeline variant', fontsize=12, fontweight='bold')
ax.set_ylabel('Reproducibility item', fontsize=12, fontweight='bold')
ax.tick_params(axis='x', labelsize=10, rotation=18, length=0)
ax.tick_params(axis='y', labelsize=10, rotation=0, length=0)
for label in ax.get_xticklabels() + ax.get_yticklabels():
    label.set_fontweight('bold')
for label in ax.get_xticklabels():
    label.set_horizontalalignment('right')

cbar = ax.collections[0].colorbar
cbar.ax.tick_params(labelsize=10, length=3, width=0.8)
cbar.outline.set_linewidth(0.8)

plt.tight_layout()

png_path = FIGURES_DIR / 'per_item_accuracy_heatmap_clean.png'
svg_path = FIGURES_DIR / 'per_item_accuracy_heatmap_clean.svg'
fig.savefig(png_path, dpi=300, bbox_inches='tight')
fig.savefig(svg_path, bbox_inches='tight')
plt.show()

print(png_path)
print(svg_path)


## Data and code reporting by year and venue (empirical papers only)

In [ ]:
score_df = pd.read_csv(SCORES_CSV)
score_df = score_df[score_df['scored'].str.lower() == 'yes'].copy()
score_df = score_df[score_df['paper_type'].str.lower() == 'empirical'].copy()
score_df = score_df[score_df['conference'].isin(['aaai', 'acl', 'neurips'])]
score_df = score_df[score_df['year'].astype(str).isin(['2023', '2024', '2025'])]

assessment_frames = []
for conf in ['aaai', 'acl', 'neurips']:
    for year in ['2023', '2024', '2025']:
        path = ROOT / 'sampled_papers' / conf / year / 'reproducibility_assessments_grobid_smart_200.csv'
        temp = pd.read_csv(path, usecols=['file_name', 'dataset_info_answer', 'code_repo_answer']).copy()
        temp['conference'] = conf
        temp['year'] = int(year)
        assessment_frames.append(temp)

assessment_df = pd.concat(assessment_frames, ignore_index=True)
merged = score_df.merge(assessment_df, on=['conference', 'year', 'file_name'], how='inner')
merged['dataset_yes'] = merged['dataset_info_answer'].str.lower().eq('yes')
merged['code_yes'] = merged['code_repo_answer'].str.lower().eq('yes')

def reporting_category(row):
    if row['dataset_yes'] and row['code_yes']:
        return 'Both code and data'
    if row['code_yes'] and not row['dataset_yes']:
        return 'Only code'
    if row['dataset_yes'] and not row['code_yes']:
        return 'Only data'
    return 'Neither'

merged['reporting_category'] = merged.apply(reporting_category, axis=1)
merged['conference_label'] = merged['conference'].map({'aaai': 'AAAI', 'acl': 'ACL', 'neurips': 'NeurIPS'})

category_order = ['Both code and data', 'Only code', 'Only data', 'Neither']
venue_order = ['AAAI', 'ACL', 'NeurIPS']
category_colors = {
    'Both code and data': '#1F4E79',
    'Only code': '#4F86B8',
    'Only data': '#9EC5DF',
    'Neither': '#C8CCD1',
}

category_counts = (
    merged.groupby(['conference_label', 'reporting_category'], as_index=False)
    .size()
    .rename(columns={'size': 'count'})
)
category_counts['conference_label'] = pd.Categorical(category_counts['conference_label'], categories=venue_order, ordered=True)
category_counts['reporting_category'] = pd.Categorical(category_counts['reporting_category'], categories=category_order, ordered=True)
category_counts = category_counts.sort_values(['conference_label', 'reporting_category'])
totals = category_counts.groupby('conference_label', as_index=False)['count'].sum().rename(columns={'count': 'total'})
category_counts = category_counts.merge(totals, on='conference_label', how='left')
category_counts['proportion'] = category_counts['count'] / category_counts['total']

year_counts = (
    merged.groupby(['conference_label', 'year', 'reporting_category'], as_index=False)
    .size()
    .rename(columns={'size': 'count'})
)
year_counts['conference_label'] = pd.Categorical(year_counts['conference_label'], categories=venue_order, ordered=True)
year_counts['reporting_category'] = pd.Categorical(year_counts['reporting_category'], categories=category_order, ordered=True)
year_counts = year_counts.sort_values(['conference_label', 'year', 'reporting_category'])
year_totals = year_counts.groupby(['conference_label', 'year'], as_index=False)['count'].sum().rename(columns={'count': 'total'})
year_counts = year_counts.merge(year_totals, on=['conference_label', 'year'], how='left')
year_counts['proportion'] = year_counts['count'] / year_counts['total']
year_counts


In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(11.6, 4.9), sharey=True)

for ax, venue in zip(axes, venue_order):
    venue_df = year_counts[year_counts['conference_label'] == venue]
    plot_df = venue_df.pivot(index='year', columns='reporting_category', values='proportion').reindex([2023, 2024, 2025])
    plot_df = plot_df[category_order]
    x_positions = range(len(plot_df.index))
    bottom = None
    for category in category_order:
        values = plot_df[category].fillna(0)
        ax.bar(
            x_positions,
            values,
            bottom=bottom,
            color=category_colors[category],
            edgecolor='white',
            linewidth=0.8,
            width=0.44,
            label=category,
        )
        bottom = values if bottom is None else bottom + values

    ax.set_title(venue, fontsize=13, pad=8, weight='semibold')
    ax.set_xticks(list(x_positions), ['2023', '2024', '2025'])
    ax.set_ylim(0, 1.0)
    ax.yaxis.set_major_locator(FixedLocator([0.0, 0.5, 1.0]))
    ax.yaxis.set_major_formatter(FormatStrFormatter('%.1f'))
    ax.minorticks_off()
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)
    ax.spines['left'].set_linewidth(1.0)
    ax.spines['bottom'].set_linewidth(1.0)
    ax.tick_params(axis='both', which='major', labelsize=10, length=4, width=1.1)
    for label in ax.get_xticklabels() + ax.get_yticklabels():
        label.set_fontweight('bold')

axes[0].set_ylabel('Proportion of empirical papers', fontsize=12, fontweight='bold')
fig.suptitle('Data and code reporting by venue and year (empirical papers only)', fontsize=16, y=1.02, weight='semibold')
handles, labels = axes[0].get_legend_handles_labels()
legend = fig.legend(handles, labels, loc='upper center', bbox_to_anchor=(0.5, -0.02), ncol=4, frameon=False, fontsize=10)
for text in legend.get_texts():
    text.set_fontweight('bold')

plt.tight_layout()
fig.subplots_adjust(bottom=0.18, top=0.82, wspace=0.18)

png_path = FIGURES_DIR / 'venue_code_data_reporting_yearwise.png'
svg_path = FIGURES_DIR / 'venue_code_data_reporting_yearwise.svg'
fig.savefig(png_path, dpi=300, bbox_inches='tight')
fig.savefig(svg_path, bbox_inches='tight')
plt.show()

print(png_path)
print(svg_path)


## Overall data and code reporting across venues (empirical papers only)

In [ ]:
plot_df = category_counts.pivot(index='conference_label', columns='reporting_category', values='proportion').reindex(venue_order)
plot_df = plot_df[category_order]
venue_totals = totals.set_index('conference_label').reindex(venue_order)['total']

fig, ax = plt.subplots(figsize=(8.2, 5.2))
bottom = None
for category in category_order:
    values = plot_df[category].fillna(0)
    ax.bar(
        plot_df.index,
        values,
        bottom=bottom,
        color=category_colors[category],
        edgecolor='white',
        linewidth=0.8,
        width=0.46,
        label=category,
    )
    bottom = values if bottom is None else bottom + values

ax.set_title('Data and code reporting across venues (empirical papers only)', fontsize=16, pad=12, weight='semibold')
ax.set_xlabel('Conference venue', fontsize=12, fontweight='bold')
ax.set_ylabel('Proportion of empirical papers', fontsize=12, fontweight='bold')
ax.set_ylim(0, 1.0)
ax.yaxis.set_major_locator(FixedLocator([0.0, 0.5, 1.0]))
ax.yaxis.set_major_formatter(FormatStrFormatter('%.1f'))
ax.minorticks_off()
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)
ax.spines['left'].set_linewidth(1.0)
ax.spines['bottom'].set_linewidth(1.0)
ax.tick_params(axis='both', which='major', labelsize=11, length=4, width=1.1)
for label in ax.get_xticklabels() + ax.get_yticklabels():
    label.set_fontweight('bold')

legend = ax.legend(loc='upper center', bbox_to_anchor=(0.5, -0.15), ncol=2, frameon=False, fontsize=10)
for text in legend.get_texts():
    text.set_fontweight('bold')

plt.tight_layout()

png_path = FIGURES_DIR / 'venue_code_data_reporting_overall.png'
svg_path = FIGURES_DIR / 'venue_code_data_reporting_overall.svg'
fig.savefig(png_path, dpi=300, bbox_inches='tight')
fig.savefig(svg_path, bbox_inches='tight')
plt.show()

print(png_path)
print(svg_path)
print(venue_totals)


## NeurIPS yearly mean reproducibility score (2016--2025)

In [ ]:
neurips_yearly = pd.read_csv(SCORES_CSV)
neurips_yearly = neurips_yearly[neurips_yearly['conference'].str.lower() == 'neurips'].copy()
neurips_yearly = neurips_yearly[neurips_yearly['scored'].str.lower() == 'yes'].copy()
neurips_yearly['year'] = neurips_yearly['year'].astype(int)
neurips_yearly['final_score'] = neurips_yearly['final_score'].astype(float)

trend_summary = neurips_yearly.groupby('year', as_index=False).agg(mean_score=('final_score', 'mean'))
trend_summary = trend_summary.sort_values('year')
neurips_overall_mean = neurips_yearly['final_score'].mean()

fig, ax = plt.subplots(figsize=(8.8, 3.6))
line_color = '#356CA5'
mean_color = '#8EC1E1'

ax.plot(
    trend_summary['year'],
    trend_summary['mean_score'],
    color=line_color,
    linewidth=2.0,
    marker='o',
    markersize=6.5,
    markerfacecolor=line_color,
    markeredgecolor='white',
    markeredgewidth=0.9,
)

ax.axhline(
    neurips_overall_mean,
    color=mean_color,
    linestyle='--',
    linewidth=2.0,
    label=f'Average {neurips_overall_mean:.3f}',
)

for _, row in trend_summary.iterrows():
    ax.text(row['year'], row['mean_score'] + 0.03, f"{row['mean_score']:.2f}", ha='center', va='bottom', fontsize=10, fontweight='bold', color=line_color)

ax.set_title('NeurIPS mean reproducibility score over time', fontsize=16, pad=10, weight='semibold')
ax.set_xlabel('Year', fontsize=12, fontweight='bold')
ax.set_ylabel('Mean normalized reproducibility score', fontsize=12, fontweight='bold')
ax.set_xlim(2015.5, 2025.5)
ax.set_ylim(0, 1.0)
ax.set_xticks(trend_summary['year'].tolist())
ax.yaxis.set_major_locator(FixedLocator([0.0, 1.0]))
ax.yaxis.set_major_formatter(FormatStrFormatter('%.1f'))
ax.minorticks_off()
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)
ax.spines['left'].set_linewidth(1.0)
ax.spines['bottom'].set_linewidth(1.0)
ax.tick_params(axis='both', which='major', labelsize=11, length=4, width=1.1)
for label in ax.get_xticklabels() + ax.get_yticklabels():
    label.set_fontweight('bold')

legend = ax.legend(loc='upper right', frameon=False, fontsize=10)
for text in legend.get_texts():
    text.set_fontweight('bold')

plt.tight_layout()

png_path = FIGURES_DIR / 'neurips_yearly_mean_repro_score_trend.png'
svg_path = FIGURES_DIR / 'neurips_yearly_mean_repro_score_trend.svg'
fig.savefig(png_path, dpi=300, bbox_inches='tight')
fig.savefig(svg_path, bbox_inches='tight')
plt.show()

print(png_path)
print(svg_path)
print(neurips_overall_mean)


## NeurIPS before vs after checklist

In [ ]:
neurips = pd.read_csv(SCORES_CSV)
neurips = neurips[neurips['scored'].str.lower() == 'yes'].copy()
neurips = neurips[neurips['conference'].str.lower() == 'neurips'].copy()
neurips['year'] = neurips['year'].astype(int)
neurips['final_score'] = neurips['final_score'].astype(float)

before_years = {2016, 2017, 2018}
after_years = {2020, 2021, 2022}
neurips = neurips[neurips['year'].isin(before_years | after_years)].copy()
neurips['period'] = neurips['year'].map(
    lambda y: 'Before checklist\n(2016-2018)' if y in before_years else 'After checklist\n(2020-2022)'
)

period_summary = neurips.groupby('period', as_index=False).agg(mean_score=('final_score', 'mean'))
period_order = ['Before checklist\n(2016-2018)', 'After checklist\n(2020-2022)']
neurips['period'] = pd.Categorical(neurips['period'], categories=period_order, ordered=True)
period_summary['period'] = pd.Categorical(period_summary['period'], categories=period_order, ordered=True)
period_summary = period_summary.sort_values('period')
period_colors = {
    'Before checklist\n(2016-2018)': '#8FAFCE',
    'After checklist\n(2020-2022)': '#7BB8A8',
}

fig, ax = plt.subplots(figsize=(8.4, 5.6))

sns.violinplot(
    data=neurips,
    x='period',
    y='final_score',
    order=period_order,
    hue='period',
    palette=period_colors,
    dodge=False,
    legend=False,
    inner=None,
    cut=0,
    linewidth=1.0,
    saturation=0.95,
    width=0.56,
    ax=ax,
)

sns.boxplot(
    data=neurips,
    x='period',
    y='final_score',
    order=period_order,
    width=0.18,
    showcaps=True,
    boxprops={'facecolor': 'white', 'edgecolor': '#111827', 'linewidth': 1.0, 'zorder': 3},
    whiskerprops={'color': '#111827', 'linewidth': 1.0},
    capprops={'color': '#111827', 'linewidth': 1.0},
    medianprops={'color': '#111827', 'linewidth': 1.2},
    showfliers=False,
    ax=ax,
)

for idx, (_, row) in enumerate(period_summary.iterrows()):
    ax.scatter(idx, row['mean_score'], color='#111827', s=28, zorder=4)

ax.set_title('NeurIPS reproducibility score before vs after checklist', fontsize=16, pad=12, weight='semibold')
ax.set_xlabel('')
ax.set_ylabel('Normalized reproducibility score', fontsize=12, fontweight='bold')
ax.set_ylim(0, 1.0)
ax.yaxis.set_major_locator(FixedLocator([0.0, 0.5, 1.0]))
ax.yaxis.set_major_formatter(FormatStrFormatter('%.1f'))
ax.minorticks_off()
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)
ax.spines['left'].set_linewidth(1.0)
ax.spines['bottom'].set_linewidth(1.0)
ax.tick_params(axis='both', which='major', labelsize=11, length=4, width=1.1)
for label in ax.get_xticklabels() + ax.get_yticklabels():
    label.set_fontweight('bold')

plt.tight_layout()

png_path = FIGURES_DIR / 'neurips_before_after_checklist_violin.png'
svg_path = FIGURES_DIR / 'neurips_before_after_checklist_violin.svg'
fig.savefig(png_path, dpi=300, bbox_inches='tight')
fig.savefig(svg_path, bbox_inches='tight')
plt.show()

print(png_path)
print(svg_path)
